In [3]:
from datasets import load_dataset

# Correct split parameter format without curly braces or spaces
ds = load_dataset("roneneldan/TinyStories", split="train[:150000]")

# Since split was specified directly, 'ds' is a Dataset instead of a DatasetDict, so we iterate over 'ds'
text = [ex["text"] for ex in ds if len(ex["text"].strip()) > 50]

In [ ]:
ds

In [4]:
import torch
import torch.nn as nn
import tiktoken
from torch.utils.data import Dataset,DataLoader
class GPTDATASET(Dataset):
  def __init__(self,text,tokenizer,max_len,stride):
    ids = tokenizer.encode(text,allowed_special="all")
    self.x, self.y = [],[]
    for i in range(0,len(ids)-max_len,stride):
      self.x.append(torch.tensor(ids[i : i+max_len]))
      self.y.append(torch.tensor(ids[i+1 : i+max_len+1]))
  def __len__(self):
    return len(self.x)
  def __getitem__(self,i):
    return self.x[i],self.y[i]

In [5]:
def make_loader(text_list, batch_size=8, max_len=256, stride=256, shuffle=True):
    # Join the list of formatted conversation strings into a single continuous string
    joined_text = "\n".join(text_list)
    tok = tiktoken.get_encoding("gpt2")
    ds = GPTDATASET(joined_text, tok, max_len, stride)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

loader = make_loader(text)
X, y = next(iter(loader))

In [6]:
class CausalSelfAttention(nn.Module):
  def __init__(self,d_model,n_heads,max_len,dropout=0.1):
    super().__init__()
    assert d_model % n_heads == 0
    self.n_heads = n_heads
    self.head_dim = d_model // n_heads
    self.qkv = nn.Linear(d_model,3*d_model,bias=False)
    self.proj = nn.Linear(d_model,d_model)
    self.drop = nn.Dropout(dropout)
    mask = torch.tril(torch.ones(max_len,max_len))
    self.register_buffer("mask",mask.view(1,1,max_len,max_len))

  def forward(self, x):
    B, T, C = x.shape

    q, k, v = self.qkv(x).chunk(3, dim=-1)

    q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)  # (B, heads, T, T)
    scores = scores.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
    weights = self.drop(torch.softmax(scores, dim=-1))

    out = weights @ v                                   # (B, heads, T, head_dim)
    out = out.transpose(1, 2).contiguous().view(B, T, C)
    return self.proj(out)

attn = CausalSelfAttention(d_model=128, n_heads=4, max_len=256)
x = torch.randn(2, 10, 128)
print(attn(x).shape)  # torch.Size([2, 10, 128])

# test 2: causality. Changing a FUTURE token must not change earlier outputs
attn.eval()
x2 = x.clone()
x2[:, 7:] = torch.randn(2, 3, 128)   # alter tokens 7..9
print(torch.allclose(attn(x)[:, :7], attn(x2)[:, :7], atol=1e-6))  # True

torch.Size([2, 10, 128])
True


In [7]:
import torch
import torch.nn as nn

class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, max_len, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size=50257, d_model=256, n_heads=4,
                 n_layers=4, max_len=256, dropout=0.1):
        super().__init__()
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(
            *[TransformerBlock(d_model, n_heads, max_len, dropout) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        x = self.blocks(x)
        logits = self.head(self.ln_f(x))

        loss = None
        if targets is not None:
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1)
            )
        return logits, loss

# tests
model = GPT()
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

x, y = next(iter(loader))
logits, loss = model(x, y)
print(logits.shape, loss.item())

16.1M params
torch.Size([8, 256, 50257]) 10.892855644226074


In [10]:
total_steps = epochs * len(train_loader)
print(f"Step {step}/{total_steps} | Train Loss: {loss.item():.3f}")

Step 8070/14624 | Train Loss: 2.288


In [11]:
torch.save(model.state_dict(), "tinystories_base_model.pt")
print("Saved checkpoint at current step successfully!")

Saved checkpoint at current step successfully!


In [8]:
import torch, tiktoken

device = "cuda" if torch.cuda.is_available() else "cpu"
print("training on:", device)


split = int(0.9 * len(text))
train_loader = make_loader(text[:split], batch_size=16, shuffle=True)
val_loader   = make_loader(text[split:], batch_size=16, shuffle=False)

model = GPT().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

@torch.no_grad()
def evaluate(loader, max_batches=20):
    model.eval()
    losses = []
    for i, (x, y) in enumerate(loader):
        if i >= max_batches: break
        _, loss = model(x.to(device), y.to(device))
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

epochs, eval_every, step = 2, 100, 0
for epoch in range(epochs):
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # prevents loss spikes
        opt.step()
        step += 1

        if step % eval_every == 0:
            print(f"step {step} | train {loss.item():.3f} | val {evaluate(val_loader):.3f}")

torch.save(model.state_dict(), "gpt_ckpt.pt")

training on: cuda
step 100 | train 4.991 | val 4.917
step 200 | train 4.332 | val 4.157
step 300 | train 4.180 | val 3.885
step 400 | train 3.914 | val 3.712
step 500 | train 3.793 | val 3.537
step 600 | train 3.553 | val 3.411
step 700 | train 3.629 | val 3.294
step 800 | train 3.508 | val 3.206
step 900 | train 3.454 | val 3.128
step 1000 | train 3.197 | val 3.055
step 1100 | train 3.145 | val 2.977
step 1200 | train 3.341 | val 2.927
step 1300 | train 3.204 | val 2.867
step 1400 | train 3.137 | val 2.818
step 1500 | train 2.818 | val 2.776
step 1600 | train 2.861 | val 2.729
step 1700 | train 2.980 | val 2.699
step 1800 | train 2.907 | val 2.667
step 1900 | train 2.827 | val 2.638
step 2000 | train 3.080 | val 2.605
step 2100 | train 2.777 | val 2.571
step 2200 | train 2.971 | val 2.541
step 2300 | train 2.822 | val 2.512
step 2400 | train 2.752 | val 2.493
step 2500 | train 2.688 | val 2.475
step 2600 | train 2.917 | val 2.451
step 2700 | train 2.735 | val 2.434
step 2800 | train 2

KeyboardInterrupt: 

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=50, temperature=0.8, top_k=40):

    model.eval()

    tok = tiktoken.get_encoding("gpt2")

    idx = torch.tensor(
        [tok.encode(prompt, allowed_special="all")],
        device=device
    )

    eos_id = tok.eot_token

    for _ in range(max_new_tokens):

        logits, _ = model(idx[:, -model.max_len:])

        logits = logits[:, -1, :] / temperature

        if top_k:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float("-inf")

        probs = torch.softmax(logits, dim=-1)

        nxt = torch.multinomial(probs, 1)

        idx = torch.cat([idx, nxt], dim=1)

        # Use the pre-resolved eos_id to avoid raising tiktoken special token exceptions
        if nxt.item() == eos_id:
            break

    return tok.decode(idx[0].tolist())

print(generate(model, "The sky was  "))